In [1]:
from datasets import Dataset
import os
import librosa
import numpy as np
from collections import defaultdict
from functools import partial
import json

from datasets import ClassLabel

import random
from datasets import Dataset, load_dataset,DatasetDict
from datasets import load_from_disk

### __1. This section converts the PD1 speech corpus into a frame-level phone classification dataset__
For each utterance, it matches the WAV audio file with its PAR annotation file, extracts acoustic features from the audio, aligns each frame with a phone label, and stores the result as a Hugging Face Dataset.

Steps to convert:

- Retrieve all files from the entire corpus.

- Locate all `.par` and `.wav` files based on their filenames and pair them up. However, it is important to note that each utterance should correspond to exactly one `.par` file and one `.wav` file. Therefore, we must also avoid duplicates.

In [2]:
# get corpus directory
workdir = "PD1"

# check whether the corpus directory exists
if not os.path.isdir(workdir):
    raise FileNotFoundError(
        f"Corpus directory not found: {os.path.abspath(workdir)}")

In [3]:
# store the WAV and PAR paths belonging to each utterance
all_paths = defaultdict(dict)

for root, _, filenames in os.walk(workdir):
    for filename in filenames:
        utt_id, ext = os.path.splitext(filename)
        ext = ext.lower() # get the extentions to collect all .wav and .par files

        # check if the folder has .wav or .par
        if ext not in {".wav", ".par"}: 
            continue

        file_path = os.path.join(root, filename)

        # detect duplicate files
        # each utterance should have exactly one .wav file and one matching .par file
        if ext in all_paths[utt_id]:
            previous_path = all_paths[utt_id][ext]
            raise ValueError(f"Duplicate {ext} file for utterance {utt_id}:\n")

        all_paths[utt_id][ext] = file_path

# keep only complete wav-and-par pairs
file_pairs = sorted([(utt_id, files[".wav"], files[".par"]) 
                     for utt_id, files in all_paths.items() 
                     if ".wav" in files and ".par" in files], 
                     key=lambda pair: pair[0])

print(f"Complete wav-and-par pairs: {len(file_pairs)}\n")

print("First five pairs:")
for pair in file_pairs[:5]:
    print(pair)

Complete wav-and-par pairs: 21587

First five pairs:
('alzn0180', 'PD1/ALZN/alzn0180.wav', 'PD1/ALZN/alzn0180.par')
('alzn0190', 'PD1/ALZN/alzn0190.wav', 'PD1/ALZN/alzn0190.par')
('alzn0200', 'PD1/ALZN/alzn0200.wav', 'PD1/ALZN/alzn0200.par')
('alzn0210', 'PD1/ALZN/alzn0210.wav', 'PD1/ALZN/alzn0210.par')
('alzn0220', 'PD1/ALZN/alzn0220.wav', 'PD1/ALZN/alzn0220.par')


- Define the required parameters. If any need to be modified later, we can simply update them in this section. 

- After defining the parameters, we choose between using Mel or Mfcc.

In [4]:
# extract Mel and Mfcc with 25ms window
def extract_mels(wav_file, sr, frame_rate, n_mels):
    y, sr = librosa.load(wav_file, sr=sr)

    hop_length = round(sr / frame_rate)
    win_length = round(0.025 * sr)

    feats = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=win_length, win_length=win_length, hop_length=hop_length, n_mels=n_mels).T 
    return feats

def extract_mfcc(wav_file, sr, frame_rate, n_mfcc, n_mels):
    y, sr = librosa.load(wav_file, sr=sr)

    hop_length = round(sr / frame_rate)
    win_length = round(0.025 * sr)

    feats = librosa.feature.mfcc(y=y, sr=sr, n_fft=win_length, win_length=win_length, hop_length=hop_length, n_mfcc=n_mfcc, n_mels=n_mels).T 
    return feats

- Reminder: choose Mel or Mfcc here

In [5]:
# define parameters
sr = 16_000 # expected sample rate, easier to compute
frame_rate = 100
n_mels = 32
n_mfcc = 13

# use Mel or Mfcc
fe = partial(extract_mels, sr=sr, frame_rate=frame_rate, n_mels=n_mels)
# fe = partial(extract_mfcc, sr=sr, frame_rate=frame_rate, n_mels=n_mels, n_mfcc =n_mfcc)

- The `.par` files contain the phone-level alignments for each utterance.

- In the following code, we read the `SAM` line to check the sample rate and read the `MAU` lines to get the start time, duration, level, and phone label.

- We also verify that the sample rate in the `.par` file matches the global defined expected sample rate used for feature extraction. If we want to change the sample rate in the future, it would be easier.

In [6]:
# read and check the alignments of the sample rate match expected sam
# read phone alignments of MAU
def read_and_check_par(par_file_path, utt_id, expected_sr=sr):
    sr_par = None
    par = []

    with open(par_file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line.startswith("SAM:"):
                sr_par = int(line.split()[1]) # get par file sample rate

            elif line.startswith("MAU:"):
                mau_feature = line.split()

                # each MAU line should contain: MAU start duration level label
                if len(mau_feature) < 5:
                    raise ValueError(
                        f"Invalid MAU line in {utt_id}: {line}"
                    )

                start = int(mau_feature[1])
                dur = int(mau_feature[2])
                level = int(mau_feature[3])
                label = " ".join(mau_feature[4:])

                par.append({
                    "start": start,
                    "dur": dur,
                    "level": level,
                    "label": label
                })
    
    # check if the sample rate in .par file matches the expected sample rate
    if sr_par == 0:
        print(
            f"Sample rate not recorded at {utt_id}, assuming sampling rate {expected_sr}")
        sr_par = expected_sr

    elif sr_par != expected_sr:
        raise ValueError(
        f"Sample rate does not match the expected sampling rate at {utt_id}: expected {expected_sr}, got {sr_par}")

    return par

- The phone alignments in the `.par` files are interval-based: each phone has a start time and a duration. However, the acoustic features are frame-based. Therefore, we convert each phone interval into individual frame labels, so every acoustic frame receives one phone label.

In [7]:
def labels_from_par_gen(par, frame_rate, sr, tier_name="phones"):
    for item in par:
        xmin = item["start"] / sr
        xmax = (item["start"] + item["dur"]) / sr

        for frame_index in range(round(xmin * frame_rate), round(xmax * frame_rate)):
            yield {"frame_index": frame_index, tier_name: item["label"]}

- In addition to acoustic features and phone labels, we also include speaker metadata. The metadata file contains information such as the original speaker ID, recording site, sex, age group, and PD2 status.

- Since the same original speaker ID may appear in different recording sites, we combine the speaker ID and recording site to create a unique speaker ID.

In [8]:
def load_speaker_metadata(metadata_file):
    speaker_metadata = {}

    with open(metadata_file, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            parts = line.split("\t")

            if len(parts) not in {5, 6}:
                raise ValueError(f"Invalid speaker metadata at line {line_number}: {line}")

            # get all the metadata by its position
            volume = parts[0]
            original_speaker_id = parts[1].strip().upper()
            recording_site = parts[2].upper()
            sex = parts[3]  # M = Male, W = Female
            age = parts[4]  # J = Jung (Young), A = Alt (Old)
            pd2 = parts[5] if len(parts) == 6 else ""

            speaker_uid = original_speaker_id + recording_site # unique speaker id

            if speaker_uid in speaker_metadata:
                raise ValueError(f"Duplicate speaker UID: {speaker_uid}")

            speaker_metadata[speaker_uid] = {
                "speaker_uid": speaker_uid,
                "speaker_orgid": original_speaker_id,
                "volume": volume,
                "recording_site": recording_site,
                "sex": sex,
                "age": age,
                "pd2": pd2,
            }

    return speaker_metadata

In [9]:
# at first we used speaker id with length of 3, but this will cause duplicate of speaker ids
# so we use speaker id with length of 4
def extract_speaker_id_from_utt_id(utt_id):
    if len(utt_id) < 4:
        raise ValueError(
            f"Invalid utterance ID: {utt_id}"
        )

    return utt_id[:4].upper()

__This part is the main dataset generator.__

- For each utterance, it extracts acoustic features from the `.wav` file, reads phone alignments from the `.par` file, and retrieves speaker metadata.

- Then, it yields one training example per frame.

- Each example contains the acoustic feature vector, frame index, utterance ID, speaker information, and the corresponding phone label.

In [10]:
# generate the dataset
def one_corpus_gen(file_pairs, fe, frame_rate, sr, tier_name="phones", speaker_metadata=None):
    for utt_id, wav_file, par_file in file_pairs:
        feats = fe(wav_file)

        par = read_and_check_par(par_file, utt_id, expected_sr=sr)

        speaker_uid = extract_speaker_id_from_utt_id(utt_id)

        if speaker_metadata is not None:
            if speaker_uid not in speaker_metadata:
                raise ValueError(
                    f"Speaker '{speaker_uid}' from utterance '{utt_id}' was not found in metadata")

            speaker_info = speaker_metadata[speaker_uid]
        else:
            speaker_info = {}

        for label_dict in labels_from_par_gen(par, frame_rate, sr, tier_name):
            frame_index = label_dict["frame_index"]

            yield {
                "feat": feats[frame_index],
                "frame_index": frame_index,
                "utt_id": utt_id,
                "speaker_uid": speaker_uid,
                "volume": speaker_info.get("volume"),
                "speaker_id": speaker_info.get("speaker_orgid"),
                "recording_site": speaker_info.get("recording_site"),
                "sex": speaker_info.get("sex"),
                "age": speaker_info.get("age"),
                "pd2": speaker_info.get("pd2"),
                "phones": label_dict["phones"]
            }

- Now we load the speaker metadata and create the Hugging Face Dataset.

- The dataset is generated frame by frame, so each row corresponds to one acoustic frame rather than one complete utterance.

- After the generation, write down the dataset.

In [11]:
metadata_file = "PD1/PD1_Doku/PD1_sprk.txt"
speaker_metadata = load_speaker_metadata(metadata_file)

dataset = Dataset.from_generator(
    one_corpus_gen,
    gen_kwargs={
        "file_pairs": file_pairs,
        "fe": fe,
        "frame_rate": frame_rate,
        "sr": sr,
        "tier_name": "phones",
        "speaker_metadata": speaker_metadata
    }
)

In [12]:
for index in range(1000, 1010):
    print(
        json.dumps(
            dataset[index],
            ensure_ascii=False,
            indent=2
        )
    )

{
  "feat": [
    0.013438409194350243,
    0.041917890310287476,
    0.07209867238998413,
    0.026714839041233063,
    0.05768875777721405,
    0.04282428324222565,
    0.06549756973981857,
    0.018665814772248268,
    0.006481272168457508,
    0.005857823882251978,
    0.002024074550718069,
    0.0028770710341632366,
    0.003712923265993595,
    0.005863086320459843,
    0.002181252231821418,
    0.0009835882810875773,
    0.0005186630878597498,
    0.00028264778666198254,
    0.00020571616187226027,
    0.0003519322781357914,
    0.00029497401556000113,
    0.0002680602192413062,
    0.00013428102829493582,
    7.232483039842919e-05,
    0.00017686655337456614,
    0.00025681365514174104,
    1.851918023021426e-05,
    2.97667406812252e-06,
    5.123007213114761e-06,
    1.119450189435156e-05,
    1.967254138435237e-05,
    2.1393008864833973e-05
  ],
  "frame_index": 58,
  "utt_id": "alzn0220",
  "speaker_uid": "ALZN",
  "volume": "2",
  "speaker_id": "ALZ",
  "recording_site": 

- This section checks whether the speakers in the generated dataset match the speakers in the metadata file.

- If a speaker appears in the metadata but not in the dataset, it may mean that this speaker has no complete WAV-PAR pairs.

- If a speaker appears in the dataset but not in the metadata, it may indicate an issue with speaker ID extraction or missing metadata.

In [13]:
metadata_file = "PD1/PD1_Doku/PD1_sprk.txt"
speaker_metadata = load_speaker_metadata(metadata_file)

metadata_speakers = set(speaker_metadata.keys())
dataset_speakers = set(dataset.unique("speaker_uid"))

print("Speakers in metadata:", len(metadata_speakers))
print("Speakers in dataset:", len(dataset_speakers))

missing_speakers = metadata_speakers - dataset_speakers
unexpected_speakers = dataset_speakers - metadata_speakers

print("Metadata speakers missing from dataset:")
print(sorted(missing_speakers))

print("Dataset speakers not found in metadata:")
print(sorted(unexpected_speakers))

Speakers in metadata: 201
Speakers in dataset: 201
Metadata speakers missing from dataset:
[]
Dataset speakers not found in metadata:
[]


### __2. This section splits the Hugging Face PD1 dataset into training, validation, and test sets.__

- The split is speaker-based, not frame-based. This means that all frames from the same speaker are assigned to only one split. This avoids speaker overlap between training, validation, and test data.

- Also, it splits the training dataset into more smaller splits to observe the model's performance under different scale of training data.

Steps to split:

- Here, we define the random seed and the split ratios.

- The seed makes the speaker shuffle reproducible.

- The test set uses 20% of the speakers, and the validation set uses 10% of the speakers.

In [14]:
seed = 42
test_ratio = 0.2
validation_ratio = 0.1

In [15]:
speaker_ids = list(dataset.unique("speaker_uid")) # get unique speaker id list and shuffle 

rand = random.Random(seed)
rand.shuffle(speaker_ids) # seed to ensure everytime shuffle the same 

n_speakers = len(speaker_ids)

# max(1, len(speaker)) to ensure minimal number of test set
n_test_speakers = max(1, round(len(speaker_ids) * test_ratio))
n_validation_speakers = max(1, round(n_speakers * validation_ratio))

# after assigning speakers to different groups, we filter the full dataset
# each frame is placed into the training, validation, or test set according to its speaker id
test_speakers = set(speaker_ids[:n_test_speakers])
validation_speakers = set(speaker_ids[n_test_speakers: n_test_speakers + n_validation_speakers])
train_speakers = set(speaker_ids[n_test_speakers + n_validation_speakers:])

train_pool = dataset.filter(lambda example: example["speaker_uid"] in train_speakers)
# not training set

validation_dataset = dataset.filter(lambda example: example["speaker_uid"] in validation_speakers)
test_dataset = dataset.filter(lambda example: example["speaker_uid"] in test_speakers)

- We print the number of speakers and samples in each split.

- This helps us check whether the split sizes look reasonable.

In [16]:
print("Training speakers:", len(train_speakers))
print("Validation speakers:", len(validation_speakers))
print("Test speakers:", len(test_speakers))
print("Training samples:", len(train_pool))
print("Validation samples:", len(validation_dataset))
print("Test samples:", len(test_dataset))

Training speakers: 141
Validation speakers: 20
Test speakers: 40
Training samples: 5570391
Validation samples: 599109
Test samples: 1515136


In [17]:
assert train_speakers.isdisjoint(test_speakers)
assert train_speakers.isdisjoint(validation_speakers)
assert validation_speakers.isdisjoint(test_speakers)

print("No speaker overlap under all sets.")

No speaker overlap under all sets.


In [18]:
train_indices = np.arange(len(train_pool))

np_rand = np.random.default_rng(seed)
np_rand.shuffle(train_indices)

In [19]:
percentages = [0.25, 0.50, 0.75, 1.00]
train_subsets = {}
frame_rate = 100

for percentage in percentages:
    n_samples = round(len(train_pool) * percentage)

    selected_indices = train_indices[:n_samples].tolist()

    subset_name = f"{int(percentage * 100)}%"
    train_subsets[subset_name] = train_pool.select(selected_indices)

    approximate_minutes = (len(train_subsets[subset_name]) / frame_rate / 60)

    print(f"{subset_name}: {len(train_subsets[subset_name])} samples, approximately {approximate_minutes:.2f} minutes")

25%: 1392598 samples, approximately 232.10 minutes
50%: 2785196 samples, approximately 464.20 minutes
75%: 4177793 samples, approximately 696.30 minutes
100%: 5570391 samples, approximately 928.40 minutes


In [20]:
mini_minutes = 10

n_mini_samples = round(mini_minutes * 60 * frame_rate)

mini_indices = train_indices[:n_mini_samples].tolist()

train_mini = train_pool.select(mini_indices)

approximate_minutes = (len(train_mini) / frame_rate / 60)

print(f"Mini training set: {len(train_mini)} samples, approximately {approximate_minutes:.2f} minutes")

Mini training set: 60000 samples, approximately 10.00 minutes


In [21]:
train_25 = train_subsets["25%"]
train_50 = train_subsets["50%"]
train_75 = train_subsets["75%"]
train_100 = train_subsets["100%"]

In [22]:
all_subsets = DatasetDict({
    "train_mini_10min": train_mini,
    "train_25": train_25,
    "train_50": train_50,
    "train_75": train_75,
    "train_100": train_100,
    "validation": validation_dataset,
    "test": test_dataset,
})

- In the following part, we check the dataset we got by printing the top 5 frames.

In [23]:
for i in range(5):
    print(all_subsets["train_25"][i])

{'feat': [7.172059122240171e-05, 1.0370530389991472e-06, 8.911578106562956e-07, 2.9801537948515033e-07, 8.87491182766098e-08, 3.129991554828848e-08, 1.6591106088981178e-07, 3.09520430619159e-07, 7.71400010535217e-08, 6.114326822626026e-08, 1.0286240836876459e-07, 8.190693989718056e-08, 2.1621715973196842e-07, 8.206690438328224e-08, 1.0636661329499475e-07, 3.078696053648855e-08, 3.049195385074199e-08, 5.6976194429125826e-08, 1.668161644374777e-07, 7.126892143105579e-08, 3.6893506916158e-08, 3.6385134905003724e-08, 3.5811577703270814e-08, 3.274183768553485e-08, 1.9121060290672176e-08, 2.2403336075171865e-08, 3.026782025017383e-08, 5.899492805383488e-08, 4.157924848868788e-08, 3.062524811525691e-08, 4.008798981658401e-08, 3.850769303426205e-08], 'frame_index': 41, 'utt_id': 'm75a4380', 'speaker_uid': 'M75A', 'volume': '3', 'speaker_id': 'M75', 'recording_site': 'A', 'sex': 'M', 'age': 'A', 'pd2': '', 'phones': '<p:>'}
{'feat': [1.0473781912878621e-05, 2.003862618948915e-06, 1.865406460410

### __3. This section encodes phone labels for model training.__
- In the previous step, the dataset still stores phone labels as strings.

- Here, we collect all observed phone labels, check them against a predefined SAMPA label list, and convert each phone label into a numeric class ID.

- The encoded dataset is then saved to disk for training.

- We collect all phone labels that appear in the full dataset. `train_100`, `validation`, and `test` together cover the complete speaker split.

- Sorting the labels makes the class order deterministic and reproducible.

In [24]:
# get all the phones
observed_phones = sorted(
    set(all_subsets["train_100"].unique("phones"))
    | set(all_subsets["validation"].unique("phones"))
    | set(all_subsets["test"].unique("phones"))) # train_100 + validation + test = whole dataset

print("Number of phone classes:", len(observed_phones))
print(observed_phones)

Number of phone classes: 44
['2:', '6', '9', '<p:>', '?', '@', 'C', 'E', 'E:', 'I', 'N', 'O', 'OY', 'S', 'U', 'Y', 'Z', 'a', 'a:', 'aI', 'aU', 'a~', 'b', 'd', 'e:', 'f', 'g', 'h', 'i:', 'j', 'k', 'l', 'm', 'n', 'o:', 'p', 'r', 's', 't', 'u:', 'v', 'x', 'y:', 'z']


__This part defines the expected phone labels.__

- The list is based on the PD1 document file: `PD1/PD1_Doku/ext_sampa.txt`

- Later, we compare this list with the labels actually observed in the dataset.

In [25]:
valid_phones = {
    # Vowels
    "a:", "a",
    "e:", "e",
    "E",
    "i:", "i", "I",
    "o:", "o", "O",
    "u:", "u", "U",
    "y:", "y", "Y",
    "E:",
    "2:", "2",
    "9",

    # Nasalized vowels
    "a~", "E~", "O~", "9~",

    # Diphthongs
    "aI", "aU", "OY",

    # Unstressed vowels
    "@", "6",

    # Consonants
    "z", "S", "Z", "C", "x", "N", "?",
    "b", "d", "f", "g", "h", "j", "k",
    "l", "m", "n", "p", "r", "s", "t", "v",
}

valid_special_labels = {
    "p:",   # silence
    "sil",  # silence
    "sp",   # silence
    "#",    # compound boundary
    "H",    # aspiration marker
    "<p:>"  # MAUS silence/pause
}

valid_exact_labels = (
    valid_phones
    | valid_special_labels
) # valid_phones + valid_special_labels

__This part is a sanity check before encoding the labels.__

- We compare the observed labels with the expected label list.

- `unknown_labels` are labels that appear in the dataset but are not included in our predefined list.

- `legal_but_absent` are expected labels that do not appear in the current dataset.

In [26]:
unknown_labels = (set(observed_phones) - valid_exact_labels)
legal_but_absent = (valid_exact_labels - set(observed_phones))

print("Number of observed labels:", len(observed_phones))
print("Number of documented exact labels:", len(valid_exact_labels))

print("\nObserved labels:")
print(sorted(observed_phones))

print("\nLabels not found in exact SAMPA list:")
print(sorted(unknown_labels))

print("\nDocumented labels absent from dataset:")
print(sorted(legal_but_absent))

Number of observed labels: 44
Number of documented exact labels: 58

Observed labels:
['2:', '6', '9', '<p:>', '?', '@', 'C', 'E', 'E:', 'I', 'N', 'O', 'OY', 'S', 'U', 'Y', 'Z', 'a', 'a:', 'aI', 'aU', 'a~', 'b', 'd', 'e:', 'f', 'g', 'h', 'i:', 'j', 'k', 'l', 'm', 'n', 'o:', 'p', 'r', 's', 't', 'u:', 'v', 'x', 'y:', 'z']

Labels not found in exact SAMPA list:
[]

Documented labels absent from dataset:
['#', '2', '9~', 'E~', 'H', 'O~', 'e', 'i', 'o', 'p:', 'sil', 'sp', 'u', 'y']


- Next, we create mappings between phone labels and numeric class IDs.

- `phone_to_id` maps each phone symbol to an integer.

- `id_to_phone` maps each integer back to its original phone symbol.

- These mappings are needed because neural network models train on numeric labels.

In [27]:
phone_to_id = {phone: class_id for class_id, phone in enumerate(observed_phones)}

id_to_phone = {class_id: phone for phone, class_id in phone_to_id.items()}

num_classes = len(observed_phones)

print(phone_to_id)

{'2:': 0, '6': 1, '9': 2, '<p:>': 3, '?': 4, '@': 5, 'C': 6, 'E': 7, 'E:': 8, 'I': 9, 'N': 10, 'O': 11, 'OY': 12, 'S': 13, 'U': 14, 'Y': 15, 'Z': 16, 'a': 17, 'a:': 18, 'aI': 19, 'aU': 20, 'a~': 21, 'b': 22, 'd': 23, 'e:': 24, 'f': 25, 'g': 26, 'h': 27, 'i:': 28, 'j': 29, 'k': 30, 'l': 31, 'm': 32, 'n': 33, 'o:': 34, 'p': 35, 'r': 36, 's': 37, 't': 38, 'u:': 39, 'v': 40, 'x': 41, 'y:': 42, 'z': 43}


- This function encodes phone labels in batches.

- For each example, it looks up the string phone label in `phone_to_id` and stores the numeric value in a new column called `label`.

In [28]:
def encode_phone_per_batch(batch):
    return {"label": [phone_to_id[phone] for phone in batch["phones"]]}

In [29]:
encoded_subsets = all_subsets.map(encode_phone_per_batch, batched=True, batch_size=10_000, desc="Encoding phone labels")

In [30]:
print(encoded_subsets["train_mini_10min"][0])

{'feat': [7.172059122240171e-05, 1.0370530389991472e-06, 8.911578106562956e-07, 2.9801537948515033e-07, 8.87491182766098e-08, 3.129991554828848e-08, 1.6591106088981178e-07, 3.09520430619159e-07, 7.71400010535217e-08, 6.114326822626026e-08, 1.0286240836876459e-07, 8.190693989718056e-08, 2.1621715973196842e-07, 8.206690438328224e-08, 1.0636661329499475e-07, 3.078696053648855e-08, 3.049195385074199e-08, 5.6976194429125826e-08, 1.668161644374777e-07, 7.126892143105579e-08, 3.6893506916158e-08, 3.6385134905003724e-08, 3.5811577703270814e-08, 3.274183768553485e-08, 1.9121060290672176e-08, 2.2403336075171865e-08, 3.026782025017383e-08, 5.899492805383488e-08, 4.157924848868788e-08, 3.062524811525691e-08, 4.008798981658401e-08, 3.850769303426205e-08], 'frame_index': 41, 'utt_id': 'm75a4380', 'speaker_uid': 'M75A', 'volume': '3', 'speaker_id': 'M75', 'recording_site': 'A', 'sex': 'M', 'age': 'A', 'pd2': '', 'phones': '<p:>', 'label': 3}


- We cast the numeric `label` column to a Hugging Face `ClassLabel`.

- This stores the label names together with the numeric IDs, making the dataset easier to interpret later.

In [31]:
phone_class_label = ClassLabel(names = observed_phones)
encoded_subsets = encoded_subsets.cast_column("label", phone_class_label)

- Finally, we check the encoded label information and the acoustic feature dimension.

- The number of classes should match the number of observed phone labels.

- The feature dimension should match the number of acoustic coefficients, for example 32 for Mel features.

In [32]:
# Check label classes and feature dimension
reference_dataset = encoded_subsets["train_100"]

label_feature = reference_dataset.features["label"]

num_classes = label_feature.num_classes
num_feat = len(reference_dataset[0]["feat"]) # e.g. num_feat = n_mel = 32, = n_mfcc = 13

print(label_feature)
print("Number of classes:", num_classes)
print("Feature dimension:", num_feat)

ClassLabel(names=['2:', '6', '9', '<p:>', '?', '@', 'C', 'E', 'E:', 'I', 'N', 'O', 'OY', 'S', 'U', 'Y', 'Z', 'a', 'a:', 'aI', 'aU', 'a~', 'b', 'd', 'e:', 'f', 'g', 'h', 'i:', 'j', 'k', 'l', 'm', 'n', 'o:', 'p', 'r', 's', 't', 'u:', 'v', 'x', 'y:', 'z'], id=None)
Number of classes: 44
Feature dimension: 32


- Reminder: choose Mel or Mfcc here.

In [33]:
# the encoded dataset is saved to disk.
encoded_subsets.save_to_disk("dataset/mel/PD1_phone_mel_subsets_encoded")
# encoded_subsets.save_to_disk("dataset/mfcc/PD1_phone_mfcc_subsets_encoded")

print("Encoded datasets saved.")

Saving the dataset (0/1 shards):   0%|          | 0/60000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1392598 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/2785196 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/4177793 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/5570391 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/599109 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1515136 [00:00<?, ? examples/s]

Encoded datasets saved.


In [34]:
print(encoded_subsets)

DatasetDict({
    train_mini_10min: Dataset({
        features: ['feat', 'frame_index', 'utt_id', 'speaker_uid', 'volume', 'speaker_id', 'recording_site', 'sex', 'age', 'pd2', 'phones', 'label'],
        num_rows: 60000
    })
    train_25: Dataset({
        features: ['feat', 'frame_index', 'utt_id', 'speaker_uid', 'volume', 'speaker_id', 'recording_site', 'sex', 'age', 'pd2', 'phones', 'label'],
        num_rows: 1392598
    })
    train_50: Dataset({
        features: ['feat', 'frame_index', 'utt_id', 'speaker_uid', 'volume', 'speaker_id', 'recording_site', 'sex', 'age', 'pd2', 'phones', 'label'],
        num_rows: 2785196
    })
    train_75: Dataset({
        features: ['feat', 'frame_index', 'utt_id', 'speaker_uid', 'volume', 'speaker_id', 'recording_site', 'sex', 'age', 'pd2', 'phones', 'label'],
        num_rows: 4177793
    })
    train_100: Dataset({
        features: ['feat', 'frame_index', 'utt_id', 'speaker_uid', 'volume', 'speaker_id', 'recording_site', 'sex', 'age', 'pd2

In [35]:
example = encoded_subsets["train_25"][0]
print(example)

{'feat': [7.172059122240171e-05, 1.0370530389991472e-06, 8.911578106562956e-07, 2.9801537948515033e-07, 8.87491182766098e-08, 3.129991554828848e-08, 1.6591106088981178e-07, 3.09520430619159e-07, 7.71400010535217e-08, 6.114326822626026e-08, 1.0286240836876459e-07, 8.190693989718056e-08, 2.1621715973196842e-07, 8.206690438328224e-08, 1.0636661329499475e-07, 3.078696053648855e-08, 3.049195385074199e-08, 5.6976194429125826e-08, 1.668161644374777e-07, 7.126892143105579e-08, 3.6893506916158e-08, 3.6385134905003724e-08, 3.5811577703270814e-08, 3.274183768553485e-08, 1.9121060290672176e-08, 2.2403336075171865e-08, 3.026782025017383e-08, 5.899492805383488e-08, 4.157924848868788e-08, 3.062524811525691e-08, 4.008798981658401e-08, 3.850769303426205e-08], 'frame_index': 41, 'utt_id': 'm75a4380', 'speaker_uid': 'M75A', 'volume': '3', 'speaker_id': 'M75', 'recording_site': 'A', 'sex': 'M', 'age': 'A', 'pd2': '', 'phones': '<p:>', 'label': 3}



### __4. This section creates site-based dataset splits for transfer experiments of the model__

- The goal is to study whether a phone classification model trained on one recording site (or dialects) can generalize to other recording sites. Different recording sites may also have different microphones, room conditions, or background noise.

- We therefore create separate train, validation, and test subsets for each recording site.

#### How to observe the model performance between different recording site?

- Each model is trained and validated on one recording site.

- The same model is then tested on all kept recording sites.

- This gives both within-site performance and cross-site transfer performance.

- train_site = A -> validation_site = A -> test_site = A/N/D

- train_site = N -> validation_site = N -> test_site = A/N/D

- train_site = D -> validation_site = D -> test_site = A/N/D

Steps:

- First, we load the encoded dataset that was previously split into `train_100`, `validation`, and `test`. These original splits are still speaker-disjoint.

- We keep this original train/validation/test structure, but further divide each split by recording site.

In [36]:
seed = 42

recording_sites = {
    "A": "Kiel",
    "B": "Bochum",
    "N": "Bonn",
    "D": "Munich",
}

source_splits = ["train_100", "validation", "test"]

# count samples for each recording site in each split
# this helps us check whether all sites have enough data for training, validation, and testing
site_counts = {}

# site_code = "A", site_name = "Kiel"
# filter the current split to select data where recording_site matches the current region code
for site_code, site_name in recording_sites.items():
    site_counts[site_code] = {}

    for split_name in source_splits:
        subset = encoded_subsets[split_name].filter(
            lambda example: example["recording_site"] == site_code
        )
        site_counts[site_code][split_name] = len(subset)

print("Sample counts by recording site:")
for site_code, site_name in recording_sites.items():
    print(
        f"{site_code} ({site_name:<7}) | "
        f"train_100: {site_counts[site_code]['train_100']:<8} "
        f"validation: {site_counts[site_code]['validation']:<8} "
        f"test: {site_counts[site_code]['test']:<8}"
    )

Sample counts by recording site:
A (Kiel   ) | train_100: 1319069  validation: 215391   test: 216520  
B (Bochum ) | train_100: 1503789  validation: 33943    test: 305856  
N (Bonn   ) | train_100: 1086887  validation: 204340   test: 610429  
D (Munich ) | train_100: 1660646  validation: 145435   test: 382331  


- Next, we find the smallest site split across the whole table. Because if one recording site has a very small subset, it may make the site-transfer experiment less reliable. 

- We noticed that one val split is too small, so we remove the entire recording site that contains the smallest cell. Because it can save much more time rather than get a "real balanced" dataset.

In [37]:
# find the global minimum among all site-split combinations
# get initialized value
min_site_code = None
min_split_name = None
min_count = None

for site_code in recording_sites:
    for split_name in source_splits:
        count = site_counts[site_code][split_name]

        if min_count is None or count < min_count:
            min_count = count
            min_site_code = site_code
            min_split_name = split_name

print(
    "Smallest cell:", f"{min_site_code} ({recording_sites[min_site_code]})", min_split_name, min_count, "samples")

Smallest cell: B (Bochum) validation 33943 samples


In [38]:
# remove the whole recording site that contains the smallest mini set
kept_recording_sites = {site_code: site_name for site_code, site_name in recording_sites.items() if site_code != min_site_code}

print("Removed recording site:")
print(min_site_code, recording_sites[min_site_code])

Removed recording site:
B Bochum


In [39]:
# extract subsets only for the kept recording sites
site_subsets = {}

for split_name in source_splits:
    site_subsets[split_name] = {}

    for site_code, site_name in kept_recording_sites.items(): # A, N, D
        subset = encoded_subsets[split_name].filter(
            lambda example: example["recording_site"] == site_code
        )# filtering specific recording site
        # A = Kiel, D = Munich, N = Bonn

        site_subsets[split_name][site_code] = subset

        print(
            f"{split_name:} "
            f"{site_code} ({site_name:}): "
            f"{len(subset)} samples"
        )

# use only the kept recording sites for site-specific transfer experiments
experiment_sites = list(kept_recording_sites.keys())
print("Experiment sites:", experiment_sites)

train_100 A (Kiel): 1319069 samples
train_100 N (Bonn): 1086887 samples
train_100 D (Munich): 1660646 samples
validation A (Kiel): 215391 samples
validation N (Bonn): 204340 samples
validation D (Munich): 145435 samples
test A (Kiel): 216520 samples
test N (Bonn): 610429 samples
test D (Munich): 382331 samples
Experiment sites: ['A', 'N', 'D']


In [40]:
# save all the data split
site_dataset_dict = DatasetDict({})

for site_code in experiment_sites:
    site_dataset_dict[f"train_{site_code}"] = site_subsets["train_100"][site_code]
    site_dataset_dict[f"validation_{site_code}"] = site_subsets["validation"][site_code]
    site_dataset_dict[f"test_{site_code}"] = site_subsets["test"][site_code]

site_dataset_dict.save_to_disk("dataset/mel_by_site/PD1_phone_mel_site_subsets_encoded")
# site_dataset_dict.save_to_disk("dataset/mfcc_by_site/PD1_phone_mfcc_site_subsets_encoded")

print("Site-based subsets saved.")
print(site_dataset_dict)

Saving the dataset (0/1 shards):   0%|          | 0/1319069 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/215391 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/216520 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1086887 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/204340 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/610429 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1660646 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/145435 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/382331 [00:00<?, ? examples/s]

Site-based subsets saved.
DatasetDict({
    train_A: Dataset({
        features: ['feat', 'frame_index', 'utt_id', 'speaker_uid', 'volume', 'speaker_id', 'recording_site', 'sex', 'age', 'pd2', 'phones', 'label'],
        num_rows: 1319069
    })
    validation_A: Dataset({
        features: ['feat', 'frame_index', 'utt_id', 'speaker_uid', 'volume', 'speaker_id', 'recording_site', 'sex', 'age', 'pd2', 'phones', 'label'],
        num_rows: 215391
    })
    test_A: Dataset({
        features: ['feat', 'frame_index', 'utt_id', 'speaker_uid', 'volume', 'speaker_id', 'recording_site', 'sex', 'age', 'pd2', 'phones', 'label'],
        num_rows: 216520
    })
    train_N: Dataset({
        features: ['feat', 'frame_index', 'utt_id', 'speaker_uid', 'volume', 'speaker_id', 'recording_site', 'sex', 'age', 'pd2', 'phones', 'label'],
        num_rows: 1086887
    })
    validation_N: Dataset({
        features: ['feat', 'frame_index', 'utt_id', 'speaker_uid', 'volume', 'speaker_id', 'recording_site